# Trening modelu segmentacji Kraken na polskich stronach EHRI

Fine-tuning domyslnego modelu segmentacji Kraken (blla) na 15 polskich stronach EHRI.

**Cel:** poprawic CER z 14,25% (domyslny segmenter) do ~11% (lepsza segmentacja).

**Metoda:**
1. Pobierz 15 polskich stron .tif + ALTO XML z EHRI
2. Podzial: 12 stron train, 3 strony validation
3. Fine-tune domyslnego modelu segmentacji z ketos segtrain --load
4. Ewaluacja: Kraken e2e z custom segmenter vs domyslny

**Uwaga:** 15 stron to malo - uzywamy fine-tuningu (nie from scratch) + augmentacji.

In [ ]:
import subprocess, sys, os
from pathlib import Path
CODE_REVISION = '4cdc2bf9edb8bd23ece7042a8e89e82a9ad34db8'
EHRI_DATASET_REPO = 'PiotrSty/ehri-dataset'
# Auto-detect environment: Kaggle, Colab, RunPod, Paperspace, Lightning AI, lub lokalnie
_WORKDIR_CANDIDATES = ('/kaggle/working', '/content', '/workspace', '/notebooks', '/teamspace/studios/this_studio')
WORKDIR = Path(next((p for p in _WORKDIR_CANDIDATES if Path(p).exists()), Path.cwd()))
repo = WORKDIR / 'OCR_engine'
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/PiotrStyla/OCR_engine.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin',CODE_REVISION],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',CODE_REVISION],check=True)
os.chdir(repo)
sys.path.insert(0,str(repo))
# Zablokowana wersja Kraken 7.1.x: API zweryfikowane przeciw 7.1.1.
# transformers jest preinstalowany na Colab i konfliktuje z kraken (wymaga
# safetensors>=0.8.0, kraken wymaga ~=0.7.0). Odinstalujemy - kraken go nie
# potrzebuje (torchmetrics importuje go tylko dla BERT score, nie uzywane).
subprocess.run([sys.executable,'-m','pip','uninstall','-y','transformers'],check=False)
subprocess.run([sys.executable,'-m','pip','install','--upgrade','kraken==7.1.1','jiwer','huggingface_hub'],check=True)
# PILLOW MUSI zostac na 11.x: torchvision (v2.PILToTensor) na Kaggle jest zbudowany
# pod starszy Pillow i z Pillow 12 przepuszcza PIL bez konwersji na tensor
# (skutkuje to 'Image object has no attribute max' w tensor_invert).
# force-reinstall naprawia tez niespojny stan dysku (C-core 11.3 vs pypi 12.3).
subprocess.run([sys.executable,'-m','pip','install','--force-reinstall','pillow==11.3.0'],check=True)
# Wyczysc cache modulow, ktore mogly byc zaimportowane przed pip install.
# PIL jest krytyczne: pip install pillow aktualizuje pliki na dysku, ale
# stare moduly PIL pozostaja w sys.modules. Nowy PIL.ImageText (z dysku)
# importuje _Ink z PIL._typing (stary, cached) -> ImportError.
for _mod in list(sys.modules):
    if _mod == 'huggingface_hub' or _mod.startswith('huggingface_hub.') \
       or _mod == 'PIL' or _mod.startswith('PIL.'):
        del sys.modules[_mod]
from importlib.metadata import version as _pkg_version
print('WORKDIR:', WORKDIR)
print('IMPORT_OK', 'kraken', _pkg_version('kraken'), 'huggingface_hub', _pkg_version('huggingface_hub'), 'pillow', _pkg_version('pillow'))
import torch, PIL
print('torch', torch.__version__, '| torchvision', _pkg_version('torchvision'), '| PIL', PIL.__version__)

In [ ]:
import torch, shutil, glob
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files
assert torch.cuda.is_available(), "GPU required; select GPU T4."
print("GPU:", torch.cuda.get_device_name(0))

ehri_dir = WORKDIR / "ehri-polish"
ehri_dir.mkdir(parents=True, exist_ok=True)
all_files = list_repo_files(EHRI_DATASET_REPO, repo_type="dataset")
polish_files = [f for f in all_files if "polish" in f and (f.endswith(".tif") or f.endswith(".xml"))]
for f in polish_files:
    path = hf_hub_download(EHRI_DATASET_REPO, f, repo_type="dataset")
    shutil.copy(path, ehri_dir / Path(f).name)

tifs = sorted(ehri_dir.glob("*.tif"))
xmls = sorted(ehri_dir.glob("*.xml"))
# Sanity check: bez tego nastepne ogniwa padaja w ciemno (pusty manifest/trening).
assert tifs, f"Brak plikow .tif po pobraniu z {EHRI_DATASET_REPO}!"
assert len(tifs) == len(xmls), f"Niesparowane strony: {len(tifs)} .tif vs {len(xmls)} .xml"
assert all(p.with_suffix('.xml').name in [x.name for x in xmls] for p in tifs), "Kazdy .tif musi miec sasiada .xml"
print(f"Polish pages: {len(tifs)}")
print(f"ALTO XML: {len(xmls)}")

kraken_model_path = hf_hub_download(EHRI_DATASET_REPO, "models/polish_nfd_9313.mlmodel", repo_type="dataset")
print("Kraken recognition model:", kraken_model_path)

In [ ]:
import random
random.seed(42)
pages = sorted(ehri_dir.glob("*.tif"))
assert len(pages) >= 3, f"Za malo stron do podzialu train/val: {len(pages)}"
random.shuffle(pages)
val_pages = pages[:3]
train_pages = pages[3:]
print(f"Train: {len(train_pages)} pages")
for p in train_pages:
    print(f"  {p.name}")
print(f"Validation: {len(val_pages)} pages")
for p in val_pages:
    print(f"  {p.name}")

train_manifest = ehri_dir / "train_manifest.txt"
val_manifest = ehri_dir / "val_manifest.txt"
# Kraken 7.x: w trybie -f xml manifest to lista sciezek do plikow XML
# (zawieraja link do obrazu + baselines).
train_manifest.write_text("\n".join(str(p.with_suffix(".xml")) for p in train_pages))
val_manifest.write_text("\n".join(str(p.with_suffix(".xml")) for p in val_pages))
print(f"\nManifests written:")
print(f"  Train: {train_manifest}")
print(f"  Val: {val_manifest}")

In [ ]:
# Znajdz domyslny model segmentacji Kraken (blla.mlmodel, wbudowany w pakiet)
import kraken
from pathlib import Path
from importlib import resources
try:
    _ref = resources.files('kraken').joinpath('blla.mlmodel')
    default_seg = str(_ref) if _ref.is_file() else None
except Exception:
    default_seg = None
if default_seg is None:
    kraken_dir = Path(kraken.__file__).parent
    cand = list(kraken_dir.rglob('blla.mlmodel')) + list(kraken_dir.rglob('*.safetensors'))
    default_seg = str(cand[0]) if cand else None
print(f'Default seg model (local): {default_seg}')

In [ ]:
# Trening: fine-tune domyslnego modelu segmentacji (Kraken 7.1)
import subprocess, shutil, glob

output_model = str(WORKDIR / 'polish_seg')
ketos_bin = shutil.which('ketos') or 'ketos'

# Kraken 7.1: -d i --workers sa globalne (przed subkomenda).
# -d WYMAGA indeksu urzadzenia 'cuda:0' - samo 'cuda' wywoluje IndexError
# w kraken.ketos.util.to_ptl_device ('cuda'.split(':') -> x[1]).
# -o to KATALOG wyjsciowy; w nim powstaja checkpoint_*.ckpt oraz
# finalny plik wag best_<score>.safetensors.
cmd = [
    ketos_bin,
    '-d', 'cuda:0',
    '--workers', '2',
    'segtrain',
    '-f', 'xml',
    '-t', str(train_manifest),
    '-e', str(val_manifest),
    '--augment',
    '-o', output_model,
    '--weights-format', 'safetensors',
    '-N', '50',
]

if default_seg:
    cmd += ['-i', default_seg, '--resize', 'new']

print('Training command:')
print(' '.join(cmd))
print()
result = subprocess.run(cmd, capture_output=True, text=True)
print('STDOUT:', result.stdout[-5000:])
print('STDERR:', result.stderr[-5000:])
print('Return code:', result.returncode)
if result.returncode != 0:
    raise RuntimeError(f'ketos segtrain zakonczyl sie bledem (rc={result.returncode}); zob. STDERR powyzej.')

# Znajdz wytrenowany model - plik lezy WEWNATRZ katalogu -o:
trained = glob.glob(output_model + '/*.safetensors') + glob.glob(output_model + '/*.mlmodel')
print('\nTrained model files:')
for t in trained:
    print(' ', t)

In [ ]:
# Ewaluacja: Kraken e2e z custom segmenter vs domyslny
# UWAGA: Kraken 7.1 - uzywamy niezdeprekowanego, wysokopoziomowego API (kraken.tasks),
# ktore wczytuje TAKZE safetensors. blla.segment()/rpred() przyjelyby tu sciezke
# zamiast obiektu modelu i wywalilyby AttributeError.
# Config z device jako stringiem 'cuda' nie przejdzie przez Lightning Fabric -
# podajemy accelerator='cuda' + device=[0] (lista indeksow GPU).
import jiwer, warnings, glob, unicodedata
from pathlib import Path
from PIL import Image
import xml.etree.ElementTree as ET
from kraken.tasks import SegmentationTaskModel, RecognitionTaskModel
from kraken.configs import SegmentationInferenceConfig, RecognitionInferenceConfig

warnings.filterwarnings('ignore')
ALTO_NS = {'alto': 'http://www.loc.gov/standards/alto/ns-v4#'}

def load_page_gt_from_alto(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []
    for tl in root.findall('.//alto:TextLine', ALTO_NS):
        text = ''
        for s in tl.findall('alto:String', ALTO_NS):
            text += s.get('CONTENT', '')
        if text:
            lines.append(text)
    return lines

output_model = str(WORKDIR / 'polish_seg')
ehri_dir = WORKDIR / 'ehri-polish'
val_xmls = [ehri_dir / p.with_suffix('.xml').name for p in val_pages]

# Znajdz wytrenowany model (lezy w katalogu -o):
trained = (glob.glob(output_model + '/*.safetensors')
           + glob.glob(output_model + '/*.mlmodel')
           + glob.glob(str(WORKDIR / '*_best.safetensors')))
import re
# ketos zostawia kilka best_<score>.safetensors - wybieramy najwyzszy score
# (glob nie gwarantuje kolejnosci, trained[0] moze byc gorszym checkpointem)
_scored = [(float(m.group(1)), p) for p in trained if (m := re.search(r'(\d+\.\d+)\.safetensors$', p))]
custom_seg = max(_scored)[1] if _scored else (trained[0] if trained else None)
print(f'Custom segmenter: {custom_seg or "BRAK - uzyjemy tylko domyslnego"}')

# Wczytaj modele raz, przed petla (wysokopoziomowe API):
recognizer = RecognitionTaskModel.load_model(kraken_model_path)
custom_model = SegmentationTaskModel.load_model(custom_seg) if custom_seg else None

DEFAULT_SEG_MODEL = SegmentationTaskModel.load_model()  # wbudowane blla.mlmodel

for seg_name, seg_task in [
    ('default', DEFAULT_SEG_MODEL),
    ('custom', custom_model),
]:
    if seg_task is None:
        print(f'\n=== {seg_name}: model not found, skipping ===')
        continue
    print(f'\n=== Kraken e2e + {seg_name} segmenter ===')
    all_refs, all_hyps = [], []
    for xml_path in val_xmls:
        page_path = xml_path.with_suffix('.tif')
        gt_lines = load_page_gt_from_alto(xml_path)
        img = Image.open(page_path).convert('L')
        seg = seg_task.predict(img, SegmentationInferenceConfig(accelerator='cuda', device=[0]))
        pred = recognizer.predict(img, seg, RecognitionInferenceConfig(accelerator='cuda', device=[0]))
        hyp_lines = [record.prediction.strip() for record in pred]
        # NFC: kraken zwraca NFD, ALTO GT jest NFC - bez normalizacji jiwer
        # liczy znaki diakrytyczne jako 2 codepointy i sztucznie zawyza metryki.
        ref = unicodedata.normalize('NFC', '\n'.join(gt_lines))
        hyp = unicodedata.normalize('NFC', '\n'.join(hyp_lines))
        all_refs.append(ref)
        all_hyps.append(hyp)
        print(f'  {page_path.name}: {len(gt_lines)} GT, {len(hyp_lines)} OCR')
    # Strona bez wierszy wygenerowalaby pusta hipoteze - usun ja z metryk
    # i pokaz wyraznie, ze segmentacja zawiodla.
    valid = [(r, h) for r, h in zip(all_refs, all_hyps) if h.strip()]
    if len(valid) < len(all_refs):
        print(f'  UWAGA: {len(all_refs) - len(valid)} stron bez wierszy OCR pominieto w metrykach')
    if valid:
        refs, hyps = zip(*valid)
        cer = jiwer.cer(list(refs), list(hyps))
        wer = jiwer.wer(list(refs), list(hyps))
        print(f'  CER: {cer:.4f}  ({cer*100:.2f}%)')
        print(f'  WER: {wer:.4f}  ({wer*100:.2f}%)')
    else:
        print('  BRAK jakichkolwiek wierszy OCR - nie liczono metryk.')

In [ ]:
# Fine-tuning modelu ROZPOZNAWANIA na 12 stronach treningowych (Kraken 7.1 ketos train).
# Uzasadnienie: eval e2e pokazal WER ~47% przy niemal idealnej liczbie wykrytych linii,
# czyli watkim gardlem jest recognizer, nie segmentacja. Start: polish_nfd_9313.mlmodel.
# -i laduje istniejacy model; --resize add rozszerza codec o brakujace znaki
# (new budowalby codec od zera i utracil pretrenowane mapowanie).
import subprocess, shutil, glob

output_model = str(WORKDIR / 'polish_nfd')
ketos_bin = shutil.which('ketos') or 'ketos'

cmd = [
    ketos_bin,
    '-d', 'cuda:0',
    '--workers', '2',
    'train',
    '-f', 'xml',
    '-t', str(train_manifest),
    '-e', str(val_manifest),
    '--augment',
    '-i', kraken_model_path,
    '--resize', 'union',
    '-o', output_model,
    '--weights-format', 'safetensors',
    '-N', '50',
]
print('Training command:')
print(' '.join(cmd))
print()
result = subprocess.run(cmd, capture_output=True, text=True)
print('STDOUT:', result.stdout[-5000:])
print('STDERR:', result.stderr[-5000:])
print('Return code:', result.returncode)
if result.returncode != 0:
    raise RuntimeError(f'ketos train zakonczyl sie bledem (rc={result.returncode}); zob. STDERR powyzej.')

trained_rec = glob.glob(output_model + '/*.safetensors') + glob.glob(output_model + '/*.mlmodel')
print('\nTrained recognizer files:')
for t in trained_rec:
    print(' ', t)

In [ ]:
# Eval e2e: custom segmenter + recognizer fine-tunowany vs oryginalny.
# Punkty odniesienia (poprzednia ewaluacja): default seg 13.45% CER / 47.40% WER,
# custom seg + oryginalny recognizer 13.12% CER / 45.50% WER.
# Komorka jest samowystarczalna: po restarcie kernela wystarcza 1, 2, 3, potem 7 i 8.
import re, jiwer, warnings, glob, unicodedata
from pathlib import Path
from PIL import Image
import xml.etree.ElementTree as ET
from kraken.tasks import SegmentationTaskModel, RecognitionTaskModel
from kraken.configs import SegmentationInferenceConfig, RecognitionInferenceConfig

warnings.filterwarnings('ignore')
ALTO_NS = {'alto': 'http://www.loc.gov/standards/alto/ns-v4#'}

def load_page_gt_from_alto(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []
    for tl in root.findall('.//alto:TextLine', ALTO_NS):
        text = ''
        for s in tl.findall('alto:String', ALTO_NS):
            text += s.get('CONTENT', '')
        if text:
            lines.append(text)
    return lines

# Custom segmenter - najwyzszy score z katalogu treningu segmentacji (komorka 5).
trained_seg = glob.glob(str(WORKDIR / 'polish_seg' / '*.safetensors'))
_scored_seg = [(float(m.group(1)), p) for p in trained_seg if (m := re.search(r'(\d+\.\d+)\.safetensors$', p))]
assert _scored_seg, 'Brak wytrenowanego segmentera - odpal najpierw komorke 5.'
seg_task = SegmentationTaskModel.load_model(max(_scored_seg)[1])
print(f'Custom segmenter: {max(_scored_seg)[1]}')

# Recognizery: oryginalny z HF oraz fine-tunowany z komorki 7 (najwyzszy score).
trained_rec = glob.glob(str(WORKDIR / 'polish_nfd' / '*.safetensors'))
_scored_rec = [(float(m.group(1)), p) for p in trained_rec if (m := re.search(r'(\d+\.\d+)\.safetensors$', p))]
assert _scored_rec, 'Brak fine-tunowanego recognizera - odpal najpierw komorke 7.'
recognizers = [
    ('original', RecognitionTaskModel.load_model(kraken_model_path)),
    ('finetuned', RecognitionTaskModel.load_model(max(_scored_rec)[1])),
]
print(f'Finetuned recognizer: {max(_scored_rec)[1]}')

ehri_dir = WORKDIR / 'ehri-polish'
val_xmls = [ehri_dir / p.with_suffix('.xml').name for p in val_pages]

for rec_name, recognizer in recognizers:
    print(f'\n=== custom segmenter + recognizer: {rec_name} ===')
    all_refs, all_hyps = [], []
    for xml_path in val_xmls:
        page_path = xml_path.with_suffix('.tif')
        gt_lines = load_page_gt_from_alto(xml_path)
        img = Image.open(page_path).convert('L')
        seg = seg_task.predict(img, SegmentationInferenceConfig(accelerator='cuda', device=[0]))
        pred = recognizer.predict(img, seg, RecognitionInferenceConfig(accelerator='cuda', device=[0]))
        hyp_lines = [record.prediction.strip() for record in pred]
        # NFC: kraken zwraca NFD, ALTO GT jest NFC - bez normalizacji jiwer
        # liczy znaki diakrytyczne jako 2 codepointy i sztucznie zawyza metryki.
        all_refs.append(unicodedata.normalize('NFC', '\n'.join(gt_lines)))
        all_hyps.append(unicodedata.normalize('NFC', '\n'.join(hyp_lines)))
        print(f'  {page_path.name}: {len(gt_lines)} GT, {len(hyp_lines)} OCR')
    valid = [(r, h) for r, h in zip(all_refs, all_hyps) if h.strip()]
    if len(valid) < len(all_refs):
        print(f'  UWAGA: {len(all_refs) - len(valid)} stron bez wierszy OCR pominieto w metrykach')
    if valid:
        refs, hyps = zip(*valid)
        cer = jiwer.cer(list(refs), list(hyps))
        wer = jiwer.wer(list(refs), list(hyps))
        print(f'  CER: {cer:.4f}  ({cer*100:.2f}%)')
        print(f'  WER: {wer:.4f}  ({wer*100:.2f}%)')
    else:
        print('  BRAK jakichkolwiek wierszy OCR - nie liczono metryk.')

In [ ]:
# Zabezpieczenie wynikow: upload najlepszego segmentera i fine-tunowanego recognizera na HF.
# Odpal zaraz po komorkach 5 i 7 - /kaggle/working ginie wraz z sesja.
# Token: sekret HF_TOKEN (Kaggle/Colab) albo interaktywny notebook_login (prawa write).
import os, re, glob
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault('HF_TOKEN', UserSecretsClient().get_secret('HF_TOKEN'))
except Exception:
    try:
        from google.colab import userdata
        os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN'))
    except Exception:
        pass
if not os.environ.get('HF_TOKEN'):
    from huggingface_hub import notebook_login
    notebook_login()
from huggingface_hub import upload_file

def best_score(pattern):
    scored = [(float(m.group(1)), p) for p in glob.glob(pattern) if (m := re.search(r'(\d+\.\d+)\.safetensors$', p))]
    return max(scored)[1] if scored else None

uploads = [
    (best_score(str(WORKDIR / 'polish_seg' / '*.safetensors')), 'models/polish_seg_best.safetensors'),
    (best_score(str(WORKDIR / 'polish_nfd' / '*.safetensors')), 'models/polish_nfd_finetuned.safetensors'),
]
for local, remote in uploads:
    if local:
        upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id='PiotrSty/ehri-dataset', repo_type='dataset')
        print(f'Uploaded {local} -> {remote}')
    else:
        print(f'Pomijam {remote} - brak pliku (uruchom komorke treningowa)')